# Bilanciamento e persistenza dei rendimenti

Proposta del 24 ottobre 2026. Dati finanziari nativi di OvernightAH e FlatComposite,
congelati nello studio precedente. Approfondimento descrittivo, non validazione
fuori campione e nessuna modifica applicativa.

La domanda è se sono cambiate frequenza delle perdite, frequenza sotto la media
storica e dipendenza nel loro susseguirsi. La soglia è la media aritmetica del
rendimento giornaliero del portafoglio nel 2000–2018, nota a fine 2018.
Non si assume il 50% dei risultati sotto media. Zeri e dati mancanti hanno
significati distinti. Le metafore non definiscono una legge probabilistica.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd
from IPython.display import display
ROOT = Path.cwd()
if not (ROOT/'analyze.py').exists():
    ROOT = Path('/home/htpc/backtrader/research/statistical_process_compatibility_20261024/balance_indicator_proposal')
sys.path.insert(0, str(ROOT))
import analyze
result = analyze.main()
summary = pd.DataFrame(result['summaries'])
view = summary[['strategy','observations','mean_daily_return','negative_fraction','positive_fraction','zero_fraction','below_mean_fraction']].copy()
view.columns = ['Strategia','Sedute','Media giornaliera (%)','Negative (%)','Positive (%)','Zero (%)','Sotto media (%)']
view.iloc[:,2:] = view.iloc[:,2:]*100
display(view.round(4))

,Strategia,Sedute,Media giornaliera (%),Negative (%),Positive (%),Zero (%),Sotto media (%)
0,OvernightAH,4779,0.1232,34.1076,47.7506,18.1419,57.1249
1,FlatComposite,4779,0.1716,41.1802,56.9784,1.8414,54.3001


## Sequenze reali sotto media

Una sequenza di 49 sedute sotto media è presente nel riferimento OvernightAH:
47 sedute hanno rendimento zero, 2 negativo. La durata va confrontata con la
storia e con la composizione della sequenza, senza chiamare 49 giorni sotto
media 49 perdite. Non viene diagnosticata qui la causa dei rendimenti a zero.

In [2]:
display(summary[['strategy','longest_negative','longest_below_mean','longest_below_start','longest_below_end','longest_below_zero_days','longest_below_negative_days']])
windows = pd.DataFrame(result['windows'])
comparison = windows[windows.window==20].copy()
comparison['Storico (%)'] = comparison.historical_fraction*100
comparison['Ipotesi iid (%)'] = comparison.naive_iid_fraction*100
display(comparison[['strategy','window','observed_windows','windows_all_below_mean','Storico (%)','Ipotesi iid (%)']].round(6))
assert all(result['validation'].values())
print('Partizioni e conteggi di tutte le finestre verificati anche con calcolo scalare indipendente.')

,strategy,longest_negative,longest_below_mean,longest_below_start,longest_below_end,longest_below_zero_days,longest_below_negative_days
0,OvernightAH,10,49,2009-02-27,2009-05-07,47,2
1,FlatComposite,10,45,2009-02-27,2009-05-01,43,2


,strategy,window,observed_windows,windows_all_below_mean,Storico (%),Ipotesi iid (%)
4,OvernightAH,20,4760,63,1.323529,0.001369
9,FlatComposite,20,4760,73,1.533613,0.000497


Partizioni e conteggi di tutte le finestre verificati anche con calcolo scalare indipendente.


## Interpretazione e proposta

Le finestre storiche si sovrappongono e non sono episodi indipendenti. Il confronto
con la formula iid mostra il rischio di supporre indipendenza; non stima una
probabilità futura esatta. La media è stimata sullo stesso riferimento: questa è
una descrizione retrospettiva, senza p-value calibrati.

Il candidato dovrebbe mostrare separatamente conteggi negativi/zero/positivi,
conteggi sotto media, lunghezze delle sequenze e transizioni. Media congelata
prima della finestra recente; riferimenti di uguale durata e ordine preservato.
Le due soglie sono legate: con media positiva ogni perdita è anche sotto media.

La capacità di riconoscere cambiamenti di frequenza e di dipendenza va misurata
separatamente, con correzione per finestre e controlli ripetuti. Le classi non
conservano l'ampiezza economica: il confronto continuo resta complementare.

[Proposta dettagliata](proposta.md). Il [NIST descrive il runs test classico](https://www.itl.nist.gov/div898/handbook/eda/section3/eda35d.htm)
per l'ordine casuale; qui si vuole confrontare con la dipendenza storica, anche
quando questa contiene già raggruppamenti. Il test classico non viene applicato
automaticamente a tale domanda.